# 🔥 TGPF Pipeline — Teaching Version

This notebook is a reconstruction exercise based on your `05_MED_00_pipeline.ipynb`.

**How to use this:**
- Each cell has an explanation of *what* and *why*, then blanks (`___`) to fill in.
- Try to complete each cell from memory or reasoning before checking the original.
- The goal is not to reproduce the code perfectly — it's to understand each decision.

**Layers covered:**
1. Python fundamentals (loops, dicts, lists, conditionals)
2. Functions — design, arguments, return values, `None` as a signal
3. pandas patterns — cumsum, groupby, merge, pivot
4. Checkpoint and error-handling architecture
5. GEE lazy evaluation and the `toBands()` insight

---
## Part 1 — Python Fundamentals

### 1A. Building a list of dictionaries

**Concept:** A *list of dictionaries* is the most common way to represent tabular-ish data before it hits pandas. Each dictionary is one row; each key is a column name.

In the pipeline, `eco_records` is built by looping over GEE features and extracting properties into dicts.

**Pattern:**
```python
result = []
for item in collection:
    result.append({
        'key': item['value'],
        ...
    })
```

**Exercise:** Given this fake feature list, build `eco_records` with keys `eco_id`, `eco_name`, `biome_name`, `geometry`.

In [2]:
# Fake data — stands in for what GEE returns
fake_features = [
    {'properties': {'ECO_ID': 701, 'ECO_NAME': 'Mediterranean conifer and mixed forests',
                    'BIOME_NUM': 5, 'BIOME_NAME': 'Temperate Conifer Forests'},
     'geometry': {'type': 'Polygon', 'coordinates': [[[28, 36], [30, 36], [30, 38], [28, 38]]]}},
    {'properties': {'ECO_ID': 789, 'ECO_NAME': 'Crete Mediterranean forests',
                    'BIOME_NUM': 12, 'BIOME_NAME': 'Mediterranean Forests, Woodlands & Scrub'},
     'geometry': {'type': 'Polygon', 'coordinates': [[[23, 35], [26, 35], [26, 36], [23, 36]]]}},
]

# TODO: Build eco_records as a list of dicts.
# Each dict should have: eco_id, eco_name, biome_num, biome_name, geometry
# (for now, just store the raw geometry dict — no ee.Geometry needed)

eco_records = []

for f in fake_features:
    p = f['properties']
    eco_records.append({
        'eco_id'    : p["ECO_ID"],
        'eco_name'  : p["ECO_NAME"],
        'biome_num' : p["BIOME_NUM"],
        'biome_name': p["BIOME_NAME"],
        'geometry'  : f["geometry"]
    })

print(f'Built {len(eco_records)} records.')
print(eco_records[0])

Built 2 records.
{'eco_id': 701, 'eco_name': 'Mediterranean conifer and mixed forests', 'biome_num': 5, 'biome_name': 'Temperate Conifer Forests', 'geometry': {'type': 'Polygon', 'coordinates': [[[28, 36], [30, 36], [30, 38], [28, 38]]]}}


In [3]:
# ANSWER — run this cell to check your work
eco_records_answer = []
for f in fake_features:
    p = f['properties']
    eco_records_answer.append({
        'eco_id'    : p['ECO_ID'],
        'eco_name'  : p['ECO_NAME'],
        'biome_num' : p['BIOME_NUM'],
        'biome_name': p['BIOME_NAME'],
        'geometry'  : f['geometry']
    })
print(eco_records_answer)

[{'eco_id': 701, 'eco_name': 'Mediterranean conifer and mixed forests', 'biome_num': 5, 'biome_name': 'Temperate Conifer Forests', 'geometry': {'type': 'Polygon', 'coordinates': [[[28, 36], [30, 36], [30, 38], [28, 38]]]}}, {'eco_id': 789, 'eco_name': 'Crete Mediterranean forests', 'biome_num': 12, 'biome_name': 'Mediterranean Forests, Woodlands & Scrub', 'geometry': {'type': 'Polygon', 'coordinates': [[[23, 35], [26, 35], [26, 36], [23, 36]]]}}]


---
### 1B. Subsetting a list with conditions

**Concept:** Two common patterns for filtering a list:
1. **Slice** — `my_list[:N]` — take the first N items
2. **List comprehension with condition** — `[x for x in my_list if x['key'] in allowed_ids]`

The pipeline supports both `TEST_N` (take first N) and `TEST_IDS` (filter by ID).

**Key Python insight:** `None` is used as a "not set" sentinel. `if TEST_IDS is not None` means "only apply this filter if the user actually set a value".

**Exercise:** Implement the subsetting logic below.

In [10]:
import pandas as pd
import numpy as np

eco_records = eco_records_answer  # use answer from above

TEST_N   = 1       # take only first 1 ecoregion
TEST_IDS = None    # no ID filter

eco_run = eco_records   # start with the full list

# TODO: if TEST_IDS is set, filter eco_run to only ecoregions whose eco_id is in TEST_IDS
if TEST_IDS is not None:
    eco_run = [e for e in eco_run if e["eco_id"] in TEST_IDS]
    print(f'Filtered to {len(eco_run)} ecoregions by ID.')

# TODO: if TEST_N is set, take only the first TEST_N ecoregions
if TEST_N is not None:
    eco_run = eco_records[:TEST_N]
    print(f'Subset to first {TEST_N} ecoregions.')

print(f'Running on {len(eco_run)} / {len(eco_records)} ecoregions.')

Subset to first 1 ecoregions.
Running on 1 / 2 ecoregions.


In [ ]:
# ANSWER
eco_run_ans = eco_records_answer
if TEST_IDS is not None:
    eco_run_ans = [e for e in eco_run_ans if e['eco_id'] in TEST_IDS]
if TEST_N is not None:
    eco_run_ans = eco_run_ans[:TEST_N]
print(eco_run_ans)

In [ ]:
celcius = [0, 20, 37, 100]

fahrenheit = [ (c * 9/5) + 32 for c in celcius]

print(fahrenheit)

[32.0, 68.0, 98.6, 212.0]


In [25]:
words = ['fire', 'rain', 'drought', 'snow', 'heat']

long_words = [w for w in words if len(w) > 4]

print(long_words)

['drought']


In [ ]:
nums = [-3, -1, 0, 4, 7, -2, 9]

pos_nums = [n for n in nums if n > 0]

print(pos_nums)

[4, 7, 9]


In [ ]:
words = ['fire', 'rain', 'drought', 'snow', 'heat']

word_length = [len(w) for w in words if len(w) > 4]
print(word_length)


[7]


In [31]:
people = [
    {'name': 'Alice', 'age': 17},
    {'name': 'Bob',   'age': 23},
    {'name': 'Carol', 'age': 15},
    {'name': 'David', 'age': 31},
]

adults = [p for p in people if p["age"] >= 18]

print(adults)

adult_names = [p["name"] for p in people if p["age"] >= 18]

print(adult_names)


[{'name': 'Bob', 'age': 23}, {'name': 'David', 'age': 31}]
['Bob', 'David']


---
## Part 2 — Functions

### 2A. Why `return None` is a design decision, not a failure

**Concept:** `compute_timing_metrics()` returns `None` when the year has too few detections. This is intentional — it's a *signal to the caller* that no valid result exists, so the caller can skip this year cleanly.

The alternative — returning an empty dict or raising an exception — would make the pipeline loop messier.

**Pattern:**
```python
def my_function(data):
    if not valid(data):
        return None          # early exit = guard clause
    # ... rest of computation
    return result

# Caller:
result = my_function(data)
if result is not None:       # only proceed if function succeeded
    do_something(result)
```

**Exercise:** Write a simplified version of `compute_timing_metrics()` that only computes `onset_doy`, `peak_doy`, and `n_detections`. Fill in the blanks.

In [ ]:
MIN_DETECTIONS  = 20
ONSET_THRESHOLD = 0.05

def compute_timing_simple(df, year):
    """
    Simplified version: only returns onset_doy, peak_doy, n_detections.
    Returns None if total detections < MIN_DETECTIONS.
    
    df   : DataFrame with columns 'doy' and 'n_detections'
    year : int
    """
    # --- Guard clause: not enough data ---
    total = df['n_detections'].___()   # TODO: sum the detections
    if total < MIN_DETECTIONS:         # should this be <, <=, >, >= ?
        return ___                     # TODO: signal failure
    
    # --- Setup ---
    df = df.copy().sort_values('doy').reset_index(drop=True)
    doys   = df['doy'].values.astype(float)
    counts = df['n_detections'].values.astype(float)
    
    # --- Cumulative fraction ---
    # cumsum() gives running total; divide by total to get fraction 0→1
    cum_frac = df['n_detections'].___() / total   # TODO
    
    # --- Onset: first DOY where cumulative fraction >= ONSET_THRESHOLD ---
    # df[cum_frac >= 0.05] gives all rows after the threshold is crossed
    # .iloc[0] takes the first such row
    onset_rows = df[cum_frac >= ___]              # TODO
    if onset_rows.empty:                          # .empty is True if no rows matched
        return None
    onset_doy = int(onset_rows.iloc[0]['doy'])
    
    # --- Peak: detection-weighted mean DOY ---
    # Think of it like a centre of mass: sum(doy * count) / sum(count)
    peak_doy = int(round( (___ * ___).sum() / ___.sum() ))  # TODO
    
    return {
        'year'        : ___,
        'onset_doy'   : ___,
        'peak_doy'    : ___,
        'n_detections': int(total),
    }

In [ ]:
# Test your function with fake data
fake_year = pd.DataFrame({
    'doy'         : list(range(180, 220)),
    'n_detections': [0,1,2,5,10,20,30,25,15,8,4,2,1,0,0,0,0,0,0,0]
})
result = compute_timing_simple(fake_year, 2010)
print(result)
# Expected: onset around DOY 182, peak around DOY 185-186

# Test the guard clause
sparse = pd.DataFrame({'doy': [180,200], 'n_detections': [3, 2]})
print('Sparse result:', compute_timing_simple(sparse, 2010))  # should print None

In [ ]:
# ANSWER
def compute_timing_simple_answer(df, year):
    total = df['n_detections'].sum()
    if total < MIN_DETECTIONS:
        return None
    df = df.copy().sort_values('doy').reset_index(drop=True)
    doys   = df['doy'].values.astype(float)
    counts = df['n_detections'].values.astype(float)
    cum_frac  = df['n_detections'].cumsum() / total
    onset_rows = df[cum_frac >= ONSET_THRESHOLD]
    if onset_rows.empty:
        return None
    onset_doy = int(onset_rows.iloc[0]['doy'])
    peak_doy  = int(round((doys * counts).sum() / counts.sum()))
    return {'year': year, 'onset_doy': onset_doy,
            'peak_doy': peak_doy, 'n_detections': int(total)}

print(compute_timing_simple_answer(fake_year, 2010))
print('Sparse:', compute_timing_simple_answer(sparse, 2010))

---
### 2B. Functions with inner functions

**Concept:** `compute_timing_metrics()` defines `doy_at_frac()` *inside* itself. This is called a **nested function** (or closure). It's useful when:
- A helper is only needed by one parent function
- The helper needs access to variables from the parent's scope (`df`, `cum_frac`) without passing them as arguments every time

The inner function can *read* `df` and `cum_frac` directly because they exist in the enclosing scope.

**Exercise:** Write `doy_at_frac()` as an inner function, and use it to compute onset, median, and end DOY.

In [ ]:
def compute_with_inner(df, year):
    total    = df['n_detections'].sum()
    df       = df.copy().sort_values('doy').reset_index(drop=True)
    cum_frac = df['n_detections'].cumsum() / total

    # TODO: define doy_at_frac as an inner function
    # It takes a single argument `frac` and returns the first DOY
    # where cum_frac >= frac, or None if no such row exists.
    def doy_at_frac(___):
        rows = df[___ >= ___]           # filter rows where cum_frac exceeds frac
        return int(rows.iloc[0]['doy']) if not rows.___ else None

    onset_doy  = doy_at_frac(___)       # 5%
    median_doy = doy_at_frac(___)       # 50%
    end_doy    = doy_at_frac(___)       # 95%

    return {'onset_doy': onset_doy, 'median_doy': median_doy, 'end_doy': end_doy}

print(compute_with_inner(fake_year, 2010))

In [ ]:
# ANSWER
def compute_with_inner_answer(df, year):
    total    = df['n_detections'].sum()
    df       = df.copy().sort_values('doy').reset_index(drop=True)
    cum_frac = df['n_detections'].cumsum() / total
    def doy_at_frac(frac):
        rows = df[cum_frac >= frac]
        return int(rows.iloc[0]['doy']) if not rows.empty else None
    return {
        'onset_doy' : doy_at_frac(0.05),
        'median_doy': doy_at_frac(0.50),
        'end_doy'   : doy_at_frac(0.95),
    }
print(compute_with_inner_answer(fake_year, 2010))

---
## Part 3 — pandas Patterns

### 3A. `cumsum()` and threshold crossing

**Concept:** `cumsum()` turns a column like `[0, 5, 10, 3]` into `[0, 5, 15, 18]` — the running total. Dividing by the total gives you a fraction from 0 to 1 that monotonically increases.

This is the core of onset detection: "at which day has 5% of the annual fire activity already occurred?"

**Exercise:** From the fake_year DataFrame, manually compute cumulative fractions and find the onset DOY.

In [ ]:
df = fake_year.copy().sort_values('doy').reset_index(drop=True)

# TODO: add a 'cumulative' column (running total of n_detections)
df['cumulative'] = df['n_detections'].___()   

# TODO: add a 'cum_frac' column (cumulative / total detections)
total = df['n_detections'].sum()
df['cum_frac'] = ___ / ___

print(df)
print()

# TODO: find onset DOY (first row where cum_frac >= 0.05)
onset_doy = int( df[df['cum_frac'] >= ___].iloc[___]['doy'] )
print(f'Onset DOY: {onset_doy}')

---
### 3B. `groupby().agg()` — summarising across groups

**Concept:** `groupby('eco_id').agg(...)` applies a summary function to each group. In `_eco_quality.csv` construction, this computes per-ecoregion statistics (CV of peak_doy, bimodal flags) across all year-rows for that ecoregion.

**Pattern:**
```python
df.groupby('group_col')['value_col'].agg(
    new_col_name = lambda x: some_function(x)
).reset_index()
```

**Exercise:** Given a fake metrics DataFrame, compute the mean and standard deviation of `peak_doy` per ecoregion.

In [ ]:
fake_metrics = pd.DataFrame({
    'eco_id'  : [701, 701, 701, 789, 789, 789],
    'year'    : [2010, 2011, 2012, 2010, 2011, 2012],
    'peak_doy': [195, 210, 188, 220, 215, 230],
    'bimodal_flag_year': [0, 1, 2, 0, 0, 1]
})

# TODO: compute mean_peak and std_peak per ecoregion
summary = (
    fake_metrics.groupby('eco_id')['peak_doy']
    .agg(
        mean_peak = ___,      # use a lambda: lambda x: round(float(x.mean()), 2)
        std_peak  = ___,
        cv_peak   = lambda x: round(float(___ / ___), 4)  # std / mean
    )
    .reset_index()
)

print(summary)

In [ ]:
# ANSWER
summary_ans = (
    fake_metrics.groupby('eco_id')['peak_doy']
    .agg(
        mean_peak = lambda x: round(float(x.mean()), 2),
        std_peak  = lambda x: round(float(x.std()),  2),
        cv_peak   = lambda x: round(float(x.std() / x.mean()), 4)
    )
    .reset_index()
)
print(summary_ans)

---
### 3C. `merge()` — joining two DataFrames on a key

**Concept:** `merge()` is like SQL JOIN. You specify the key column(s) and the join type.

In the pipeline, ecoregion-level quality metrics (one row per eco) are merged onto the per-year metrics (many rows per eco) — this is a *many-to-one* join, and it broadcasts the quality values across all year-rows for each ecoregion.

**Exercise:** Merge `fake_metrics` with `summary_ans` so every year-row also has `mean_peak` and `cv_peak`.

In [ ]:
# TODO: merge fake_metrics with summary_ans on 'eco_id', using a left join
# 'left' join means: keep all rows from fake_metrics, add matching columns from summary_ans
master = fake_metrics.___(summary_ans, on=___, how=___)

print(master)

In [ ]:
# ANSWER
master_ans = fake_metrics.merge(summary_ans, on='eco_id', how='left')
print(master_ans)

---
### 3D. `pivot_table()` — reshaping from long to wide

**Concept:** The interannual profile correlation requires a *years × doy* matrix — one row per year, one column per day. But daily counts are stored in *long* format (one row per eco-year-doy).

`pivot_table()` reshapes long → wide:
```python
df.pivot_table(index='year', columns='doy', values='n_detections', fill_value=0)
```

**Exercise:** Create a fake long-format daily DataFrame and pivot it.

In [ ]:
# Fake long-format daily data for one ecoregion, 3 years, 5 DOYs
fake_daily = pd.DataFrame({
    'year'        : [2010]*5 + [2011]*5 + [2012]*5,
    'doy'         : [180, 181, 182, 183, 184] * 3,
    'n_detections': [0, 2, 5, 3, 1,
                     1, 4, 8, 6, 2,
                     0, 1, 3, 9, 4]
})

# TODO: pivot to years × doy matrix
pivot = fake_daily.pivot_table(
    index   = ___,
    columns = ___,
    values  = ___,
    fill_value = ___
)

print(pivot)
print('Shape:', pivot.shape)  # should be (3, 5)

---
## Part 4 — Checkpoint and Error-Handling Architecture

### 4A. The checkpoint pattern

**Concept:** A checkpoint means: "if this output file already exists on disk, skip the computation and load the result instead."

This is essential for long-running pipelines. If a 6-hour run crashes at hour 5, you don't want to restart from zero.

**Pattern:**
```python
if os.path.exists(output_path):
    # Load cached result
    result = pd.read_csv(output_path)
else:
    # Do the expensive computation
    result = expensive_function()
    result.to_csv(output_path, index=False)
```

**Why the daily file is the checkpoint signal, not the metrics file:**
The metrics file may not exist if all years had insufficient detections. The daily counts file always gets written. So the pipeline checks for the daily file — its presence means the ecoregion was fully processed.

**Exercise:** Write checkpoint logic for a single ecoregion.

In [ ]:
import os

# Simulate paths
output_dir = '/tmp/tgpf_test'
os.makedirs(output_dir, exist_ok=True)

eco_id   = 701
eco_name = 'Mediterranean_conifer_and_mixed_forests'

daily_path  = os.path.join(output_dir, f'{eco_id}_{eco_name}_daily.csv')
metrics_path = os.path.join(output_dir, f'{eco_id}_{eco_name}.csv')

# TODO: write the checkpoint logic
# If daily_path exists AND metrics_path exists → load metrics, print 'Skipping'
# If daily_path exists but metrics_path does NOT → print 'Recovering from daily'
# If neither exists → print 'Running GEE fetch'

if ___:
    if ___:
        print('Skipping — already done.')
    else:
        print('Recovering metrics from daily file.')
else:
    print('Running GEE fetch.')

---
### 4B. `try/except` — catching errors without crashing

**Concept:** GEE calls can fail for many reasons (rate limits, network issues, invalid geometries). Without `try/except`, one failure would crash the entire pipeline.

**Pattern:**
```python
try:
    result = risky_function()
except Exception as e:
    # e contains the error message
    print(f'Failed: {e}')
    log_failure(e)
    continue  # skip to next iteration in a loop
```

**Key insight:** `continue` inside a loop means "skip the rest of this iteration and move to the next one" — it's how you skip a failed year without stopping the whole pipeline.

**Exercise:** Simulate a loop where some GEE calls fail and log the failures.

In [ ]:
def fake_gee_call(year):
    """Simulates a GEE call that fails on certain years."""
    if year in [2007, 2015]:
        raise ConnectionError(f'GEE rate limit exceeded for year {year}')
    return pd.DataFrame({'doy': range(1, 6), 'n_detections': [0, 5, 10, 3, 1]})

YEARS        = list(range(2005, 2011))
failed_years = []
results      = []

for year in YEARS:
    # TODO: wrap the fake_gee_call in try/except
    # On success: append {'year': year, 'rows': len(df)} to results
    # On failure: append {'year': year, 'reason': str(e)} to failed_years, then continue
    ___:
        df = fake_gee_call(year)
        results.append({'year': year, 'rows': len(df)})
    ___ Exception ___ e:
        failed_years.append({'year': ___, 'reason': str(___)})
        print(f'  {year}: ERROR — {e}')
        ___  # skip to next year

print('\nSuccessful:', results)
print('Failed:    ', failed_years)

In [ ]:
# ANSWER
failed_years_ans = []
results_ans      = []
for year in YEARS:
    try:
        df = fake_gee_call(year)
        results_ans.append({'year': year, 'rows': len(df)})
    except Exception as e:
        failed_years_ans.append({'year': year, 'reason': str(e)})
        print(f'  {year}: ERROR — {e}')
        continue
print('\nSuccessful:', results_ans)
print('Failed:    ', failed_years_ans)

---
## Part 5 — GEE Concepts

### 5A. Lazy evaluation — what GEE actually does

**Concept:** When you write GEE code like `terra.filterDate(start, end)`, nothing is computed yet. GEE builds a *description* of what you want — a computation graph. The computation only runs when you call `.getInfo()` or trigger an export.

This is called **lazy evaluation**. It's the fundamental mental model for understanding why GEE behaves the way it does.

**Analogy:** Writing a GEE expression is like writing a recipe. `.getInfo()` is like actually cooking it.

**Why this matters for your pipeline:**
- Calling `.getInfo()` inside a `.map()` function will fail — you're trying to cook inside the recipe.
- The `toBands()` trick works because it builds one big recipe (365 bands stacked) and cooks it once.

**Reflection questions (no code — think through and write your answers):**

**Q1:** Why does the original slow version (calling `reduceRegion` per day in a loop) hit 'Too many concurrent aggregations'?

*Your answer:*

---

**Q2:** `toBands()` collapses a 365-image collection into a single 365-band image. Why does this allow one `reduceRegion` call to return all 365 daily counts?

*Your answer:*

---

**Q3:** The band names come out as `'0_day_001'`, `'1_day_002'`, etc. (with an index prefix added by `toBands()`). The pipeline uses `band_name[-3:]` to parse the DOY. Why is this potentially fragile?

*Your answer:*

---
### 5B. The `toBands()` pattern in pure Python

**Concept:** Even without GEE, you can understand the *structure* of the `toBands()` approach. The idea is:
1. Build N items (one per day) — each is a function of a day offset
2. Stack them all together into one object
3. Process the whole stack in one operation

In Python, the equivalent is building a list of Series (one per day) and concatenating them.

**Exercise:** Build a `year_matrix` DataFrame using a loop over days — mimicking the structure of `get_daily_counts()`.

In [ ]:
import datetime
import calendar

# Fake 'pixel' data: one row per (date, pixel)
# In GEE, this is the fire detection raster. Here it's just a dict.
fake_detections = {
    datetime.date(2010, 7, 1): 5,
    datetime.date(2010, 7, 2): 12,
    datetime.date(2010, 7, 4): 3,
    datetime.date(2010, 8, 15): 8,
}

year = 2010
n_days = 366 if calendar.isleap(year) else 365

# TODO: build a list of dicts, one per day of the year
# Each dict: {'doy': <1-indexed day>, 'n_detections': <count from fake_detections or 0>}
rows = []
for d in range(___, ___):   # d = 0 to n_days-1
    date = datetime.date(year, 1, 1) + datetime.timedelta(days=___)
    doy  = ___ + 1           # 1-indexed
    count = fake_detections.get(___, ___)   # 0 if date not in dict
    rows.append({'doy': doy, 'n_detections': count})

df_year = pd.DataFrame(rows)
print(f'Rows: {len(df_year)}')
print(df_year[df_year['n_detections'] > 0])   # show only active days

In [ ]:
# ANSWER
rows_ans = []
for d in range(0, n_days):
    date  = datetime.date(year, 1, 1) + datetime.timedelta(days=d)
    doy   = d + 1
    count = fake_detections.get(date, 0)
    rows_ans.append({'doy': doy, 'n_detections': count})
df_ans = pd.DataFrame(rows_ans)
print(df_ans[df_ans['n_detections'] > 0])

---
## Part 6 — Integration Exercise

### Put it all together: mini-pipeline

**Exercise:** Without looking at the original, write a mini version of the main pipeline loop.

Given:
- `eco_run`: list of ecoregion dicts (use the fake ones from earlier)
- `YEARS`: `[2010, 2011, 2012]`
- `compute_timing_simple()`: the function you wrote in 2A
- `fake_gee_call()`: returns a daily DataFrame (use the one from 4B)

Your loop should:
1. Loop over ecoregions
2. For each ecoregion, loop over years
3. Call `fake_gee_call(year)` inside `try/except`
4. Call `compute_timing_simple()` on the result
5. If metrics are not None, append to `all_metrics` with `eco_id` added
6. Print progress

In [ ]:
YEARS_MINI = [2010, 2011, 2012]
all_metrics = []
failed      = []

for eco in ___:               # loop over ecoregions
    eco_id   = ___
    eco_name = ___
    print(f'\n=== {eco_name} (ID: {eco_id}) ===')

    for year in ___:          # loop over years
        try:
            df = fake_gee_call(___)
        except Exception as e:
            failed.append({'eco_id': eco_id, 'year': year, 'reason': str(e)})
            print(f'  {year}: ERROR — {e}')
            ___               # skip to next year

        metrics = compute_timing_simple(___, ___)

        if metrics is ___:    # insufficient detections
            print(f'  {year}: skipped (no metrics)')
            continue

        metrics['eco_id'] = ___
        all_metrics.___(metrics)
        print(f'  {year}: onset={metrics["onset_doy"]}, peak={metrics["peak_doy"]}')

print(f'\nTotal metric rows: {len(all_metrics)}')
print(f'Failed years:      {len(failed)}')
print(pd.DataFrame(all_metrics))

---
## Reflection Checklist

After completing all exercises, you should be able to answer these without looking anything up:

- [ ] What does `list.append()` do, and when would you use `list.extend()` instead?
- [ ] What is a guard clause, and why is returning `None` better than raising an exception in this pipeline?
- [ ] What does `cumsum()` return, and what does dividing by the total give you?
- [ ] What is the difference between `.agg()` and `.apply()` in pandas?
- [ ] What is a left merge, and why does it matter here?
- [ ] What does `os.path.exists()` do, and why is the daily file the checkpoint signal?
- [ ] What does `continue` do inside a for loop?
- [ ] What is lazy evaluation in GEE, and why can't you call `.getInfo()` inside `.map()`?
- [ ] Why is `toBands()` + one `reduceRegion` faster than 365 separate `reduceRegion` calls?

If you can explain all of these clearly (even to a rubber duck), you genuinely own the code.